In [ ]:
!pip install openai-whisper
!pip install langchain
!pip install -U langchain langchain-community
!pip install python-dotenv
!pip install openai
!pip install pinecone
!pip install -U langchain-pinecone
!pip install streamlit

  Using cached openai_whisper-20250625-py3-none-any.whl
  Using cached more_itertools-10.8.0-py3-none-any.whl.metadata (39 kB)
  Using cached numba-0.60.0-cp39-cp39-win_amd64.whl.metadata (2.8 kB)
  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl.metadata (59 kB)
  Using cached tiktoken-0.12.0-cp39-cp39-win_amd64.whl.metadata (6.9 kB)
  Using cached torch-2.8.0-cp39-cp39-win_amd64.whl.metadata (30 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached llvmlite-0.43.0-cp39-cp39-win_amd64.whl.metadata (4.9 kB)
  Using cached regex-2026.1.15-cp39-cp39-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.5-cp39-cp39-win_amd64.whl.metadata (39 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.m

In [2]:
import os
import subprocess

def convert_video_to_audio(video_path, audio_path, ffmpeg_dir=r"C:\FFmpeg\ffmpeg-8.0.1-essentials_build\bin"):
    """
    Converts a video file to WAV audio using FFmpeg.

    Parameters:
    - video_path (str): Path to input video file
    - audio_path (str): Path to output audio file
    - ffmpeg_dir (str): Directory containing ffmpeg.exe
    """
    # Ensure FFmpeg is in PATH
    os.environ["PATH"] += os.pathsep + ffmpeg_dir
    ffmpeg_exe = os.path.join(ffmpeg_dir, "ffmpeg.exe")

    # Run FFmpeg command
    subprocess.run([
        ffmpeg_exe,
        "-y",          # overwrite if file exists
        "-i", video_path,
        "-ac", "1",    # mono channel
        "-ar", "16000",# 16 kHz sample rate
        audio_path
    ], check=True)

# -----------------------------
# Example usage
# -----------------------------
video_file = "news.mp4"
audio_file = "news.wav"

convert_video_to_audio(video_file, audio_file)
print(f"Audio extracted to {audio_file}")

Audio extracted to news.wav


In [3]:
import whisper

def transcribe_audio(audio_file, model_size="tiny", output_txt=None):
    """
    Transcribes an audio file using OpenAI Whisper and optionally writes the transcript to a text file.

    Parameters:
    - audio_file (str): Path to the audio file (WAV, MP3, etc.)
    - model_size (str): Whisper model size to use ('tiny', 'base', 'small', 'medium', 'large')
    - output_txt (str or None): Path to save the transcript as a text file. If None, no file is written.

    Returns:
    - str: Transcribed text
    """
    # Load the Whisper model
    model = whisper.load_model(model_size)
    
    # Transcribe the audio
    result = model.transcribe(audio_file)
    transcript = result["text"]
    
    # Write to text file if a path is provided
    if output_txt:
        with open(output_txt, "w", encoding="utf-8") as f:
            f.write(transcript)
    
    return transcript

# -----------------------------
# Example usage
# -----------------------------
# audio_file = "video_audio.wav"
output_txt_file = "transcript.txt"

transcript = transcribe_audio(audio_file, output_txt=output_txt_file)
print("Transcript:\n", transcript)
print(f"\nTranscript saved to: {output_txt_file}")

c:\Ironhack\Capstone 1\.venv\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcript:
  Well, let's discuss these latest developments further now with Wolfgang Posti, who joins us from Vienna. He's a security policy and defense analyst. Here in the studio in Doha, we have Muhammad al-Masri, Professor at the Doha Institute for Graduate Studies, and Al-Jazeera's Resourced Arhuz reported extensively from Iran. Resour, let me start with you, in fact. Why is Iran attacking goal-sates energy infrastructure and how is it gaming the outcome? Well, for Iranians, I think they have been quite clear. Now, what they're implementing is what they have been seeing in the last couple of weeks. So again, again, we have heard from the Iran officials that in case of the war they're going to do two things. First, they're going to raise the course for Washington, and to force Washington to think it twice. They were hoping that Washington is going to reassess his position and seeing that the war is going to be extremely critical for them and maybe changing their posture. And secon

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = splitter.split_text(transcript)


In [ ]:
# app_streamlit.py

import os
from dotenv import load_dotenv
import streamlit as st
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# Load API keys
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

st.set_page_config(page_title="Video Q&A", layout="centered")

st.title("🎬 Video Q&A App")
st.write("Ask questions about your video transcript.")

# Assume vector_db already exists
# vector_db = LC_PineconeVectorStore(...)

retriever = vector_db.as_retriever(search_type="similarity")

qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(
        model_name="gpt-3.5-turbo",
        openai_api_key=OPENAI_API_KEY
    ),
    chain_type="stuff",
    retriever=retriever
)

question = st.text_input("Enter your question")

if st.button("Ask"):

    if not question.strip():
        st.warning("Please enter a question.")
    else:
        with st.spinner("Thinking..."):
            answer = qa_chain.run(question)

        st.success(answer)

c:\Ironhack\Capstone 1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All vectors in Pinecone index cleared.


C:\Users\Friedrich\AppData\Local\Temp\ipykernel_25416\3673513014.py:39: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)


In [6]:
from langchain_pinecone import PineconeVectorStore
vector_db = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    text_key="text"
)
# Create embeddings for all chunks at once
chunk_embeddings = embeddings.embed_documents(chunks)

# Prepare vectors to upsert
vectors_to_upsert = [
    {"id": str(i), "values": vector, "metadata": {"text": chunk}}
    for i, (chunk, vector) in enumerate(zip(chunks, chunk_embeddings))
]

# Upsert into Pinecone
vector_db.index.upsert(vectors=vectors_to_upsert)

{'upserted_count': 19}

In [7]:

retriever = vector_db.as_retriever(search_type="similarity")

from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model_name="gpt-3.5-turbo", openai_api_key=OPENAI_API_KEY),
    chain_type="stuff",
    retriever=retriever
)

answer = qa_chain.run("what did iran attack according to the transcript?")
print("Answer:", answer)

C:\Users\Friedrich\AppData\Local\Temp\ipykernel_25416\1925532371.py:7: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm=ChatOpenAI(model_name="gpt-3.5-turbo", openai_api_key=OPENAI_API_KEY),
C:\Users\Friedrich\AppData\Local\Temp\ipykernel_25416\1925532371.py:12: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  answer = qa_chain.run("what did iran attack according to the transcript?")


Answer: According to the transcript, Iran attacked energy infrastructure in Gulf states.


In [12]:
!pip install streamlit

  Using cached streamlit-1.50.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached altair-5.5.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-6.2.6-py3-none-any.whl.metadata (5.6 kB)
  Using cached protobuf-6.33.5-cp39-cp39-win_amd64.whl.metadata (593 bytes)
  Using cached pyarrow-21.0.0-cp39-cp39-win_amd64.whl.metadata (3.4 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.2-py3-none-any.whl.metadata (4.3 kB)
Using cached streamlit-1.50.0-py3-none-any.whl (10.1 MB)
Using cached altair-5.5.0-py3-none-any.whl (731 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached cachetools-6.2.6-py3-no

In [ ]:
# app_gradio.py

import os
from dotenv import load_dotenv
import gradio as gr
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# Load environment variables
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Create retriever
retriever = vector_db.as_retriever(search_type="similarity")

# Create QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(
        model_name="gpt-3.5-turbo",
        openai_api_key=OPENAI_API_KEY
    ),
    chain_type="stuff",
    retriever=retriever
)

# Function for answering questions
def answer_question(question):
    if not question.strip():
        return "Please enter a question."

    answer = qa_chain.run(question)
    return answer


# Gradio UI
with gr.Blocks() as demo:

    gr.Markdown("# 🎬 Video Q&A App")
    gr.Markdown("Ask questions about your video transcript.")

    question = gr.Textbox(
        label="Enter your question",
        placeholder="What is the video about?"
    )

    answer = gr.Textbox(label="Answer")

    ask_btn = gr.Button("Ask")

    ask_btn.click(
        fn=answer_question,
        inputs=question,
        outputs=answer
    )

demo.launch()

ModuleNotFoundError: No module named 'pkg_resources'